# Regression 

## Setup

### Imports

In [2]:
import os
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display

### Load Datasets

In [5]:
base_dir = os.path.join(os.getcwd(), '..')
processed_data_dir = os.path.join(base_dir, 'processed_data')

open_meteo_filepath = os.path.join(processed_data_dir, 'open_meteo_processed_data.csv')
open_meteo_df = pd.read_csv(open_meteo_filepath, parse_dates=['datetime'])

ausgrid_filepath = os.path.join(processed_data_dir, 'ausgrid_processed_data.csv')
ausgrid_df = pd.read_csv(ausgrid_filepath, parse_dates=['datetime'])

### Training-Testing Dataset Creation

We will create two testing datasets. The first will evaluate model performance on unseen customers, using models trained to predict PV generation generally rather than for specific customers. It will hold out all three years of data for a subset of customers.

The second will evaluate model performance on unseen days for models trained separately for each customer. Its goal is to predict future generation for known customers. This dataset uses a single chronological split, so it may not include data from all seasons.

In [8]:
unseen_customer_testing_split = 0.15
unseen_datetime_testing_split = 0.15

First, we merse the smart-meter data and weather data on `datetime`.

In [7]:
data = ausgrid_df.merge(open_meteo_df, on='datetime', how='left').sort_values('datetime')

First, we choose a subset of the customers to hold out. We want to choose a group of customers whose combined data is near the target dataset size specified by `data`. `cumulative_customer_rows` is the running total of rows for shuffled customers. This will allow us to determine how many customers to hold out so their combined rows are closest to the target test-set size.

In [9]:
rng = np.random.default_rng(42)

customer_counts = data.groupby("Customer", sort=False).size()
shuffled_customers = rng.permutation(customer_counts.index.to_numpy())
customer_counts_shuffled = customer_counts.loc[shuffled_customers]
cumulative_customer_rows = customer_counts_shuffled.cumsum()

display(cumulative_customer_rows.rename("Cumulative rows").to_frame())

,Cumulative rows
Customer,
119,52608
106,105216
185,157824
144,210432
66,263040
...,...
252,15568080
20,15620688
193,15673296


In [ ]:
customer_target_rows = unseen_customer_testing_split * len(data)
customer_count_end = (cumulative_customer_rows - customer_target_rows).abs().argmin() + 1
customer_test_customers = shuffled_customers[:customer_count_end]